In [ ]:
# --- CELL 0: DOWNLOAD DATASET (WORKS ON LOCAL) ---
import kagglehub

# Download dataset
path = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("Dataset path:", path)

# --- IMPORTS ---
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- PARAMETERS ---
IMG_SIZE = 64
BASE_PATH = os.path.join(path, "cell_images")

# --- LOAD DATA ---
def load_dataset(base_path):
    images, labels = [], []

    for label, folder in enumerate(['Uninfected', 'Parasitized']):
        folder_path = os.path.join(base_path, folder)

        for f in os.listdir(folder_path):
            if f.endswith('.png'):
                img = cv2.imread(os.path.join(folder_path, f))
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(label)

    return np.array(images), np.array(labels)

X, y = load_dataset(BASE_PATH)

# Normalize
X = X / 255.0

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# --- DATA AUGMENTATION ---
datagen = ImageDataGenerator(
    rotation_range=25,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train)

# --- MODEL ---
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

from tensorflow.keras.metrics import Recall

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', Recall()]
)

model.summary()

# --- CALLBACKS ---
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_model11.h5",
    monitor='val_accuracy',
    save_best_only=True
)

# --- TRAIN ---
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, checkpoint]
)


# --- AUTO THRESHOLD OPTIMIZATION ---
y_probs = model.predict(X_test)

thresholds = np.arange(0.2, 0.6, 0.02)

best_threshold = 0.5
best_score = float('inf')

best_cm = None

for t in thresholds:
    y_temp = (y_probs > t).astype(int)
    cm = confusion_matrix(y_test, y_temp)

    fp = cm[0][1]
    fn = cm[1][0]

    # Custom objective: prioritize FN, but also control FP
    score = fn * 2 + fp   # FN weighted more

    if score < best_score:
        best_score = score
        best_threshold = t
        best_cm = cm

# Apply best threshold
y_pred = (y_probs > best_threshold).astype(int)

print(f"Optimal Threshold Found: {best_threshold}")

print("\nConfusion Matrix at Optimal Threshold:\n", best_cm)

print("\nCNN Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# =========================
# VISUALIZATION SECTION
# =========================

# --- 1. ACCURACY & LOSS GRAPH ---
plt.figure()
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Train', 'Validation'])
plt.show()

plt.figure()
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])
plt.show()

# --- 2. CONFUSION MATRIX ---
cm = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Uninfected', 'Parasitized'],
            yticklabels=['Uninfected', 'Parasitized'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# --- 3. SAMPLE PREDICTIONS ---
def plot_sample_predictions(X, y_true, y_pred, num=9):
    plt.figure(figsize=(8,8))
    for i in range(num):
        plt.subplot(3,3,i+1)
        plt.imshow(X[i])
        plt.title(f"T:{y_true[i]} P:{y_pred[i][0]}")
        plt.axis('off')
    plt.show()

plot_sample_predictions(X_test, y_test, y_pred)

# --- 4. CLASS DISTRIBUTION ---
plt.figure()
unique, counts = np.unique(y, return_counts=True)
plt.bar(['Uninfected', 'Parasitized'], counts)
plt.title("Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()